# Frozen runtime contract

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from pathlib import Path
import hashlib,json,os,re,sys
ROOT=TRACE_ROOT
CODE=ROOT/'assets/upstream_source'
REFERENCE=ROOT/'outputs/reference'
PROV=ROOT/'evidence/provenance'
PYTHON=str(ROOT/'outputs/environments/runtime-verified/bin/python')
if os.environ.get('TRACE_LAB_RUNTIME_PYTHON'): PYTHON=os.environ['TRACE_LAB_RUNTIME_PYTHON']
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):Path(path).write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')
def verify_inputs():
    out={}
    for manifest in [ROOT/'upstream/source_manifest.json',PROV/'source_build_manifest.json',PROV/'model_data_manifest.json',PROV/'dataset_manifest.json']:
        for row in read_json(manifest):
            actual=sha(CODE/row['path'])
            if actual!=row['sha256']:raise ValueError('Frozen input changed: '+row['path'])
            out[row['path']]=actual
    for name in ['preference_scoring','baseline_rewards','environment_adapter','telemetry_recording','unity_runtime','policy_training']:
        book=read_json(ROOT/'notebooks/library'/f'{name}.ipynb')
        code='\n'.join(''.join(c['source']) for c in book['cells'] if 'module' in c.get('metadata',{}).get('tags',[]))
        out['notebook_source/'+name]=hashlib.sha256(code.encode()).hexdigest()
    return out
def worker_command(module,args):
    bootstrap=''.join(read_json(ROOT/'notebooks/library/notebook_imports.ipynb')['cells'][2]['source'])
    script=bootstrap+'\nimport '+module+'\nraise SystemExit('+module+'.main())'
    return [PYTHON,'-B','-c',script,*map(str,args)]
def resolve(config_path,attempt='attempt-01',worker=176):
    if not re.fullmatch(r'attempt-\d{2}',attempt):raise ValueError('Expected attempt-NN')
    if type(worker) is not int or not 0<=worker<1000:raise ValueError('Invalid worker ID')
    path=Path(config_path);cfg=read_json(path)
    allowed={'checkpoint_check':('TASK',4096,1100,2100,2),'task_baseline':('TASK',51200,1101,2101,10),'max_support_baseline':('MAX',51200,1102,2102,10),'random_baseline':('RANDOM',0,None,None,10)}
    condition,budget,policy_seed,sim_seed,episodes=allowed[cfg['run_id']]
    if cfg['condition']!=condition or cfg['budget_decisions']!=budget or cfg['evaluation']['episodes']!=episodes or cfg['evaluation_episodes']!=episodes:raise ValueError('Development budget/condition mismatch')
    if condition!='RANDOM':
        if cfg['training_decisions']!=budget or cfg['ppo']['n_steps']!=2048 or budget%2048 or cfg['ppo']['device']!='cpu' or cfg['ppo']['seed']!=policy_seed or cfg['simulator_initialization_seed']!=sim_seed:raise ValueError('Frozen training contract mismatch')
    expected={'window_decisions':15,'horizon_decisions':600,'window_seconds':3.0,'horizon_seconds':120.0,'first_valid_pair_seconds':6.0,'action_nvec':[3,3],'observation_schema':'solid81_time_qincrease_valid_fresh_age_v1','headless':True,'environments':1,'time_scale':1,'capture_fps':5,'torch_num_threads':1,'torch_num_interop_threads':1}
    for key,value in expected.items():
        if cfg.get(key)!=value:raise ValueError('Frozen interface mismatch: '+key)
    expected_seeds=[3091,3092] if cfg['run_id']=='checkpoint_check' else list(range(3001,3011))
    if cfg['evaluation']['requested_simulator_initialization_seeds']!=expected_seeds or not cfg['evaluation']['fresh_process_per_requested_simulator_seed']:raise ValueError('Evaluation seed/process contract mismatch')
    if condition!='RANDOM':
        if cfg['policy_initialization_seed']!=policy_seed or cfg['policy']!='MlpPolicy' or not cfg['evaluation']['deterministic_policy_actions']:raise ValueError('Policy/evaluation contract mismatch')
        expected_ppo={'learning_rate':.0003,'n_steps':2048,'batch_size':64,'n_epochs':10,'gamma':.99,'gae_lambda':.95,'clip_range':.2,'ent_coef':0.,'vf_coef':.5,'max_grad_norm':.5,'device':'cpu','seed':policy_seed,'policy_kwargs':{'net_arch':{'pi':[64,64],'vf':[64,64]},'activation_fn':'torch.nn.Tanh','ortho_init':True,'optimizer_class':'torch.optim.Adam','optimizer_kwargs':{'eps':1e-5}}}
        for key,value in expected_ppo.items():
            if cfg['ppo'].get(key)!=value:raise ValueError('Frozen PPO mismatch: '+key)
        if cfg['checkpoint_interval_decisions']!=(2048 if cfg['run_id']=='checkpoint_check' else 10240):raise ValueError('Checkpoint interval mismatch')
    elif cfg['evaluation']['random_action_sampler_seeds']!=list(range(4001,4011)):raise ValueError('Random action seeds mismatch')
    cfg.update(attempt_id=attempt,worker_id=worker,input_hashes=verify_inputs(),planned_config_sha256=sha(path),output_directory=str(ROOT/'outputs/runs'/cfg['run_id']/attempt))
    return cfg

def reserve_attempt(run_root,attempt,restart_of=None):
    """Do not touch a completed/previous attempt; only an explicit failed restart is eligible."""
    if not re.fullmatch(r"attempt-\d{2}",attempt):raise ValueError("Expected attempt-NN")
    if restart_of is not None and not re.fullmatch(r"attempt-\d{2}",restart_of):raise ValueError("Invalid restart identity")
    run_root=Path(run_root)
    if (run_root/'COMPLETE.json').exists():raise FileExistsError('Completed run ID already exists: '+str(run_root))
    attempts=sorted(p for p in run_root.glob('attempt-*') if p.is_dir()) if run_root.exists() else []
    if attempts:
        if restart_of is None:raise FileExistsError('Run ID already has an attempt; no implicit resume/retry')
        if len(attempts)!=1 or attempts[0].name!=restart_of or attempt==restart_of:
            raise ValueError('Only one explicit fresh restart of the failed first attempt is allowed')
        result=read_json(attempts[0]/'result.json')
        if result.get('status')!='FAIL':raise ValueError('Previous attempt is not a recorded failure')
    elif restart_of is not None:raise ValueError('Restart reference does not exist')
    if attempts:
        with (run_root/'.restart_claim').open('x') as claim:claim.write(attempt+'\n')
    else:
        run_root.mkdir(parents=True,exist_ok=False)
    target=run_root/attempt;target.mkdir(exist_ok=False)
    return target
print('Frozen runtime contract definitions/execution completed.')


Frozen runtime contract definitions/execution completed.


In [3]:
print('Frozen input identities:', len(verify_inputs()))

Frozen input identities: 280
